In [23]:
# Import necessary libraries
import pandas as pd
import numpy as np
import requests
import json
import time

In [45]:
# Load in site water level data dictionary
with open('/capstone/aridgw/data/site_data/site_coords.json', 'r') as f:
    site_coords = {k: tuple(v) for k, v in json.load(f).items()}

# Load in clean water level data csv that will be merged to
clean_site_data = "/capstone/aridgw/data/site_data/clean_site_waterdata_04222026.csv"
df_sites = pd.read_csv(clean_site_data)

In [ ]:
# Use API to access OpenET Data
# header = {"Authorization": "31WmFjhG0zzzyqT7LyXfuTS9sHopnzU90kxnLRFSPAuwE8w4N7kg6zuPeBRs"}
# Personal New API Key
# header = {"Authorization": "9lj4XCBC0Fnie8YfNXcIezrWnUN6nPlZGu5NKOvyx6icYAxwqrJ7UzXOcArZ"}
# Henry API Key
#header = {"Authorization": "IeZA48KSBbWoU3a50ybYlaA77Qeal3H2UKLAA86KYJ9UKw94rXmcHKhxEOTg"}
# Marie API Key 
header = {"Authorization": "1PpvkVvVazNkYjtGAPbcQ3IswdwsAOQGx3Ev4ONyACuXUUAx25zaorWf0tg0"}
# Set sites query parameter to site coordinates dictionary
sites = site_coords

# Select boundary box side length
size_km = 2

# Extract data sections less than 10 years as required by the API
date_ranges = [
    ("2000-01-01", "2009-12-31", 2.0), 
    ("2010-01-01", "2019-12-31", 2.0),
    ("2020-01-01", "2025-12-31", 2.1), # Use version 2.1 for newer versions which may not have accurate data for older versions
]

dfs = [] # Create empty dataframe

# Create function to generate boundary box given coordinates and boundary box size
def make_square_geometry(lat, lon, size_km):
    half = size_km / 2
    lat_offset = half / 111.0
    lon_offset = half / (111.0 * np.cos(np.radians(lat)))
    return [
        lon - lon_offset, lat + lat_offset,
        lon - lon_offset, lat - lat_offset,
        lon + lon_offset, lat - lat_offset,
        lon + lon_offset, lat + lat_offset,
        lon - lon_offset, lat + lat_offset,
    ]

# Retry failed requests to handle intermittent server errors
def query_with_retry(header, polygon_args, retries=3, wait=5):
    for attempt in range(retries):
        resp = requests.post(
            headers=header,
            json=polygon_args,
            url="https://openet-api.org/raster/timeseries/polygon"
        )
        if resp.status_code == 200:
            return resp
        print(f"    Attempt {attempt + 1} failed — retrying in {wait}s...")
        time.sleep(wait)
    return resp

# Use API to extract ET data 
for name, (lat, lon) in sites.items():
    print(f"Querying {name}...")
    
    for start_date, end_date, version in date_ranges:

        polygon_args = { # Define query parameters
            "date_range": [start_date, end_date],
            "interval": "monthly",
            "geometry": make_square_geometry(lat, lon, size_km),
            "model": "Ensemble",
            "reducer": "mean",
            "variable": "ET",
            "reference_et": "gridMET",
            "units": "mm",
            "file_format": "JSON",
            "version": version
        }

        resp = query_with_retry(header, polygon_args) # Request polygon API with retry logic

        print(f"  {name} {start_date}–{end_date} — Status: {resp.status_code}") # Report status of data to console

        if resp.status_code != 200: # Return error if unable to extract data
            print(f"  ERROR: {resp.json()}")
            continue

        data = resp.json()
        df = pd.DataFrame(data)

        # Add columns to dataframe to specify site specific information
        df["site"] = name
        df["latitude"] = lat
        df["longitude"] = lon
        df["bbox_side"] = size_km
        df["open_et_version"] = version
        dfs.append(df)

df_monthly = pd.concat(dfs, ignore_index=True) # Add new data as new observations in dataframe
print(df_monthly.head())

Querying KSGS.371852100505801...
    Attempt 1 failed — retrying in 5s...
    Attempt 2 failed — retrying in 5s...
    Attempt 3 failed — retrying in 5s...
  KSGS.371852100505801 2020-01-01–2024-12-31 — Status: 404
  ERROR: {'detail': 'Invalid geometry, cannot be parsed. Please check formatting.'}
Querying KSGS.372043101363101...
    Attempt 1 failed — retrying in 5s...


KeyboardInterrupt: 

In [9]:
# Load piece csv with et extraction, delete later
et_2000_2019_path = "/Users/richardmonteslemus/capstone/GW-depth-data-exploration/ET_pre_data/pre_openet_data_monthly_2km.csv"
et_2000_2019 = pd.read_csv(et_2000_2019_path)

et_2020_2024_path = "/Users/richardmonteslemus/capstone/GW-depth-data-exploration/ET_pre_data/et_2020_2024.csv"
et_2020_2024 = pd.read_csv(et_2020_2024_path)

et_2025_path = "/Users/richardmonteslemus/capstone/GW-depth-data-exploration/ET_pre_data/et_2025.csv"
et_2025 = pd.read_csv(et_2025_path)

et_missing_fills_1_path = "/Users/richardmonteslemus/capstone/GW-depth-data-exploration/ET_pre_data/et_missing_fills_1.csv"
et_missing_fills_1= pd.read_csv(et_missing_fills_1_path)

et_missing_fills_2_path = "/Users/richardmonteslemus/capstone/GW-depth-data-exploration/ET_pre_data/et_missing_fills_2.csv"
et_missing_fills_2= pd.read_csv(et_missing_fills_2_path)

pre_df_monthly = pd.concat([et_2000_2019, et_2020_2024, et_2025, et_missing_fills_1, et_missing_fills_2 ], ignore_index=True)


In [ ]:
# pre_df_monthly.drop_duplicates(inplace=True)

In [ ]:
# pre_df_monthly

,time,et,site,latitude,longitude,bbox_side,open_et_version
0,2000-01-01,11.852,KSGS.371852100505801,37.31502,-100.8505,2,2.0
1,2000-02-01,28.081,KSGS.371852100505801,37.31502,-100.8505,2,2.0
2,2000-03-01,42.692,KSGS.371852100505801,37.31502,-100.8505,2,2.0
3,2000-04-01,58.962,KSGS.371852100505801,37.31502,-100.8505,2,2.0
4,2000-05-01,106.906,KSGS.371852100505801,37.31502,-100.8505,2,2.0
...,...,...,...,...,...,...,...
16435,2024-08-01,25.175,mcmullen_valley,33.95725,-113.0800,2,2.1
16436,2024-09-01,14.519,mcmullen_valley,33.95725,-113.0800,2,2.1
16437,2024-10-01,7.187,mcmullen_valley,33.95725,-113.0800,2,2.1
16438,2024-11-01,3.790,mcmullen_valley,33.95725,-113.0800,2,2.1


In [ ]:
# Find sites with incomplete data
# pre_df_monthly.groupby('site').size().reset_index(name='count').query('count < 312').ungroup()

,site,count


In [ ]:

#pre_df_monthly["time"] = pd.to_datetime(pre_df_monthly["time"], format='%Y-%m-%d')
# pre_df_monthly["year"] = pre_df_monthly["time"].dt.year
# pre_df_monthly


,time,et,site,latitude,longitude,bbox_side,open_et_version,year
0,2000-01-01,11.852,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
1,2000-02-01,28.081,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
2,2000-03-01,42.692,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
3,2000-04-01,58.962,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
4,2000-05-01,106.906,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
...,...,...,...,...,...,...,...,...
16435,2024-08-01,25.175,mcmullen_valley,33.95725,-113.0800,2,2.1,2024
16436,2024-09-01,14.519,mcmullen_valley,33.95725,-113.0800,2,2.1,2024
16437,2024-10-01,7.187,mcmullen_valley,33.95725,-113.0800,2,2.1,2024
16438,2024-11-01,3.790,mcmullen_valley,33.95725,-113.0800,2,2.1,2024


In [ ]:
# pre_df_monthly.groupby('year')['time'].max()

year
2000   2000-12-01
2001   2001-12-01
2002   2002-12-01
2003   2003-12-01
2004   2004-12-01
2005   2005-12-01
2006   2006-12-01
2007   2007-12-01
2008   2008-12-01
2009   2009-12-01
2010   2010-12-01
2011   2011-12-01
2012   2012-12-01
2013   2013-12-01
2014   2014-12-01
2015   2015-12-01
2016   2016-12-01
2017   2017-12-01
2018   2018-12-01
2019   2019-12-01
2020   2020-12-01
2021   2021-12-01
2022   2022-12-01
2023   2023-12-01
2024   2024-12-01
2025   2025-12-01
Name: time, dtype: datetime64[ns]

In [ ]:
# pre_df_monthly.groupby('year')['site'].nunique()

year
2000    50
2001    50
2002    50
2003    50
2004    50
2005    50
2006    50
2007    50
2008    50
2009    50
2010    50
2011    50
2012    50
2013    50
2014    50
2015    50
2016    50
2017    50
2018    50
2019    50
2020    50
2021    50
2022    50
2023    50
2024    50
2025    50
Name: site, dtype: int64

In [ ]:
# See all dates missing for a site
# site_dates = pre_df_monthly[pre_df_monthly['site'] == 'mcmullen_valley']['time'].unique()
# full_dates = pd.date_range('2000-01', '2024-12', freq='MS').strftime('%Y-%m-%d')
# missing = set(full_dates) - set(site_dates)
# print(sorted(missing))

['2020-01-01', '2020-02-01', '2020-03-01', '2020-04-01', '2020-05-01', '2020-06-01', '2020-07-01', '2020-08-01', '2020-09-01', '2020-10-01', '2020-11-01', '2020-12-01', '2021-01-01', '2021-02-01', '2021-03-01', '2021-04-01', '2021-05-01', '2021-06-01', '2021-07-01', '2021-08-01', '2021-09-01', '2021-10-01', '2021-11-01', '2021-12-01', '2022-01-01', '2022-02-01', '2022-03-01', '2022-04-01', '2022-05-01', '2022-06-01', '2022-07-01', '2022-08-01', '2022-09-01', '2022-10-01', '2022-11-01', '2022-12-01', '2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01', '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01', '2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01', '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01']


In [ ]:
# Cold for saving old pre_openet_data
# df_monthly.to_csv(f"ET_pre_data/pre_openet_data_monthly_{size_km}km.csv", index=False)

In [27]:
df_monthly

,time,et,site,latitude,longitude,bbox_side,open_et_version,year
0,2000-01-01,11.852,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
1,2000-02-01,28.081,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
2,2000-03-01,42.692,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
3,2000-04-01,58.962,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
4,2000-05-01,106.906,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
...,...,...,...,...,...,...,...,...
16435,2024-08-01,25.175,mcmullen_valley,33.95725,-113.0800,2,2.1,2024
16436,2024-09-01,14.519,mcmullen_valley,33.95725,-113.0800,2,2.1,2024
16437,2024-10-01,7.187,mcmullen_valley,33.95725,-113.0800,2,2.1,2024
16438,2024-11-01,3.790,mcmullen_valley,33.95725,-113.0800,2,2.1,2024


In [28]:
# Convert time column to datetime, extract year
df_monthly["time"] = pd.to_datetime(df_monthly["time"], format='%Y-%m-%d')
df_monthly["year"] = df_monthly["time"].dt.year
df_monthly = df_monthly.rename(columns={"et": "monthly_et_avg"})
df_monthly

,time,monthly_et_avg,site,latitude,longitude,bbox_side,open_et_version,year
0,2000-01-01,11.852,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
1,2000-02-01,28.081,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
2,2000-03-01,42.692,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
3,2000-04-01,58.962,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
4,2000-05-01,106.906,KSGS.371852100505801,37.31502,-100.8505,2,2.0,2000
...,...,...,...,...,...,...,...,...
16435,2024-08-01,25.175,mcmullen_valley,33.95725,-113.0800,2,2.1,2024
16436,2024-09-01,14.519,mcmullen_valley,33.95725,-113.0800,2,2.1,2024
16437,2024-10-01,7.187,mcmullen_valley,33.95725,-113.0800,2,2.1,2024
16438,2024-11-01,3.790,mcmullen_valley,33.95725,-113.0800,2,2.1,2024


In [33]:
# Convert monthly ET data into csv
df_monthly.to_csv(f"open_et_data/openet_data_monthly_{size_km}km.csv", index=False)

In [34]:
# Check number of monthly et observations per site
# Ensure value consistency
df_monthly.groupby(["site"]).size().reset_index(name="n") 

,site,n
0,KSGS.371852100505801,312
1,KSGS.372043101363101,312
2,KSGS.372539100142504,312
3,KSGS.373331098033301,312
4,KSGS.373607100565301,312
5,KSGS.374111099070401,312
6,KSGS.374125100344101,312
7,KSGS.374747100552101,312
8,KSGS.375145100485701,312
9,KSGS.375454101075401,312


In [35]:
# Calculate a single monthly average per year 
df_annual = (
    df_monthly
    .groupby(["site", "year", "latitude", "longitude", "bbox_side", "open_et_version"])['monthly_et_avg']
    .mean()
    .reset_index()
    .rename(columns={"monthly_et_avg": "annual_et_avg"})
)

In [36]:
df_annual

,site,year,latitude,longitude,bbox_side,open_et_version,annual_et_avg
0,KSGS.371852100505801,2000,37.31502,-100.8505,2,2.0,60.401833
1,KSGS.371852100505801,2001,37.31502,-100.8505,2,2.0,63.970500
2,KSGS.371852100505801,2002,37.31502,-100.8505,2,2.0,64.001917
3,KSGS.371852100505801,2003,37.31502,-100.8505,2,2.0,65.875583
4,KSGS.371852100505801,2004,37.31502,-100.8505,2,2.0,61.988250
...,...,...,...,...,...,...,...
1295,willcox,2021,32.03619,-109.7540,2,2.1,55.621833
1296,willcox,2022,32.03619,-109.7540,2,2.1,67.415833
1297,willcox,2023,32.03619,-109.7540,2,2.1,70.138000
1298,willcox,2024,32.03619,-109.7540,2,2.1,68.251250


In [39]:
# Rescale annual et 
df_annual["scaled_annual_et_avg"] = (df_annual["annual_et_avg"] * 12).round(3)
df_annual = df_annual.drop(columns=["annual_et_avg"])


In [40]:
# Check number of scaled_annual_et_avg values per year
# Ensure value consistency
df_annual.groupby(["year"]).size().reset_index(name="n") 

,year,n
0,2000,50
1,2001,50
2,2002,50
3,2003,50
4,2004,50
5,2005,50
6,2006,50
7,2007,50
8,2008,50
9,2009,50


In [41]:
# Convert annual ET data into csv
df_annual.to_csv(f"open_et_data/openet_data_annual_{size_km}km.csv", index=False)

In [42]:
# Remove lat and long to above columns duplicates to prepare for merge 
df_annual = df_annual.drop(columns=["latitude", "longitude"])

In [43]:
df_annual

,site,year,bbox_side,open_et_version,scaled_annual_et_avg
0,KSGS.371852100505801,2000,2,2.0,724.822
1,KSGS.371852100505801,2001,2,2.0,767.646
2,KSGS.371852100505801,2002,2,2.0,768.023
3,KSGS.371852100505801,2003,2,2.0,790.507
4,KSGS.371852100505801,2004,2,2.0,743.859
...,...,...,...,...,...
1295,willcox,2021,2,2.1,667.462
1296,willcox,2022,2,2.1,808.990
1297,willcox,2023,2,2.1,841.656
1298,willcox,2024,2,2.1,819.015


In [46]:
df_sites.groupby(["site_id"]).size().reset_index(name="n") 

,site_id,n
0,KSGS.371852100505801,21
1,KSGS.372043101363101,21
2,KSGS.372539100142504,21
3,KSGS.373331098033301,21
4,KSGS.373607100565301,21
5,KSGS.374111099070401,21
6,KSGS.374125100344101,21
7,KSGS.374747100552101,21
8,KSGS.375145100485701,21
9,KSGS.375454101075401,21


In [47]:
# Merge ET data to water level data 
df_merged_et = pd.merge(
    df_sites,
    df_annual,
    left_on=['site_id', 'year_value'],
    right_on=['site', 'year'],
    how='outer'
)

In [48]:
# Remove rows containing NA in columns important for analysis
df_merged_et = df_merged_et.dropna(subset=["depth_to_gw_m", "scaled_annual_et_avg"])

In [49]:
# Change year_value to int
df_merged_et["year_value"] = df_merged_et["year_value"].astype(int)

In [50]:
# Check number of observations per site in merged df
df_merged_et.groupby(["site_id"]).size().reset_index(name="n")

,site_id,n
0,KSGS.371852100505801,21
1,KSGS.372043101363101,21
2,KSGS.372539100142504,21
3,KSGS.373331098033301,21
4,KSGS.373607100565301,21
5,KSGS.374111099070401,21
6,KSGS.374125100344101,21
7,KSGS.374747100552101,21
8,KSGS.375145100485701,21
9,KSGS.375454101075401,21


In [51]:
# Remove duplicate year column
df_merged_et = df_merged_et.drop(columns='year')
df_merged_et = df_merged_et.drop(columns='site')

In [52]:
# Turn merged df into csv
# Convert monthly ET data into csv
df_merged_et.to_csv(f"open_et_data/merged_openet_data_{size_km}km.csv", index=False)